<a href="https://colab.research.google.com/github/zhenxuekai-lab/llm-learning-journey/blob/main/day01_llm_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

print("Model loaded!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded!


In [ ]:
text = "I love machine learning."

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print("Original text:")
print(text)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

Original text:
I love machine learning.

Tokens:
['I', 'Ġlove', 'Ġmachine', 'Ġlearning', '.']

Token IDs:
[40, 2948, 5662, 6832, 13]


In [ ]:
texts =[
    "I love machine learning.",
    "我喜欢机器学习。",
    "Large language models are powerful",
    "大型语言模型很强大"
]

for text in texts:
  tokens = tokenizer.tokenize(text)
  ids = tokenizer.encode(text)

  print("=" * 50)
  print("Text:", text)
  print("Tokens:", tokens)
  print("Number of tokens:", len(ids))

Text: I love machine learning.
Tokens: ['I', 'Ġlove', 'Ġmachine', 'Ġlearning', '.']
Number of tokens: 5
Text: 我喜欢机器学习。
Tokens: ['æĪĳåĸľæ¬¢', 'æľºåĻ¨', 'åŃ¦ä¹ł', 'ãĢĤ']
Number of tokens: 4
Text: Large language models are powerful
Tokens: ['Large', 'Ġlanguage', 'Ġmodels', 'Ġare', 'Ġpowerful']
Number of tokens: 5
Text: 大型语言模型很强大
Tokens: ['å¤§åŀĭ', 'è¯Ńè¨Ģ', 'æ¨¡åŀĭ', 'å¾Ī', 'å¼ºå¤§']
Number of tokens: 5


In [ ]:
messages =[
    {
        "role": "user",
        "content": "Explain what a large language model is in one sentence."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=False
)

generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(answer)

A large language model refers to an artificial intelligence system that can generate human-like text with high accuracy and natural-sounding speech, capable of performing tasks such as summarizing complex information, writing creative content, or even composing complete sentences.


In [ ]:
prompt = "The capital of France is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
  outputs = model(**inputs)
  logits = outputs.logits

next_token_logits = logits[0, -1, :]

probabilities = torch.softmax(
    next_token_logits,
    dim=-1
)

top_probs, top_ids = torch.topk(
    probabilities,
    20
)

for prob, token_id in zip(top_probs, top_ids):
  token = tokenizer.decode([token_id])

  print(
      repr(token),
      f"{prob.item()*100:.2f}%"
  )

' Paris' 29.49%
' ______' 12.30%
':\n' 6.59%
':\n\n' 5.81%
' located' 4.81%
' __' 4.81%
' ____' 4.25%
' the' 3.52%
' (' 2.43%
' [' 2.28%
':' 1.89%
' _____' 1.66%
'\n' 1.47%
' a' 1.22%
'____' 1.15%
' in' 1.01%
'\n\n' 0.89%
' called' 0.84%
'.\n' 0.58%
' ___' 0.54%


In [ ]:
def generate_with_temperature(temp):

  messages = [
      {
          "role": "user",
          "content":"Give me a creative name for an AI healthcare startup."
      }
  ]

  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )

  inputs = tokenizer(
      text,
      return_tensors="pt"
  ).to(model.device)

  outputs = model.generate(
      **inputs,
      max_new_tokens=60,
      do_sample=True,
      temperature=temp,
      top_p=0.9
  )

  result = outputs[0][inputs["input_ids"].shape[-1]:]

  return tokenizer.decode(
      result,
      skip_special_tokens=True
  )

In [ ]:
print("Temperature = 0.2")
print(generate_with_temperature(0.2))

print("\nTemperature = 0.8")
print(generate_with_temperature(0.8))

print("\nTemperature = 1.3")
print(generate_with_temperature(1.3))

Temperature = 0.2
"HealthiSphere"

Temperature = 0.8
Sure! Here's a creative name suggestion for your AI healthcare startup:

**"HealthiTech: The Smart Health Solutions Company"**

This name incorporates several key elements:
- **"HealthiTech"**: Symbolizes the integration of technology into traditional healthcare.
- **"The Smart Health Solutions Company

Temperature = 1.3
How about "NeuroHealth.ai"? This name combines the medical field with innovative technology to highlight our unique approach and capabilities in neurology and health care. It's a nod to the brain and neurological systems that we aim to explore and improve using cutting-edge AI algorithms and medical data. It sounds both
